In [1]:
##### IMPORTS

import os
os.environ['picaso_refdata'] = r'C:\Users\Alex\Desktop\Picaso\picaso\reference' # THIS MUST GO BEFORE YOUR IMPORT STATEMENT
os.environ['PYSYN_CDBS'] = r'C:\Users\Alex\Desktop\Picaso\grp\redcat\trds' # This is for the stellar data discussed below.

# General
import gc
import numpy as np
import astropy.units as u
import bd_support as sup

from pathlib import Path
from itertools import product

# Picaso and Virga
from picaso import justdoit as jdi
from virga import justdoit as vj

# To see what clouds are availible
# vj.available()

In [2]:
##### CONFIGURATIONS

# Directories
sonor_path  = r'C:\Users\Alex\Desktop\Picaso\data\bob' # Sonora db
# sonor_path  = '/groups/tkaralidi/pbraunschweig/training_set/profiles/'
virga_path  = r'C:\Users\Alex\Desktop\Picaso\data\virga'  # Virga
opaci_path  = None # Opacity db
# opaci_path  = r'D:\opacity_500k_for_R5000_egpoutput.db'
output_path = Path(r'C:\Users\Alex\Desktop\Picaso\NN_project\compare_cases')
# output_path = 'home/al864695/ouputs'

# Constant values
wav_range   = [0.5, 30.0] # microns
MH          = 1.0        # [M/H] metallicity factor ~ solar
MU          = 2.36       # Average MU
# R           = 300        # resolution
R           = 5000       # resolution

# Dictionary for cloud naming conventions
cloud_dict  = {'Fe': '1', 'KCl': '2', 'Mg2SiO4': '3', 'MgSiO3': '4', 'Na2S': '5'}

In [3]:
##### METHOD BROWN DWARF SPECTRUM (PC VERSION)


def bd_spectrum(Teff, logg, fsed, kzz):
    """
    Compute a BD emission spectrum with Virga clouds.
    """
    # Opacity & inputs
    #  opa    = jdi.opannection(wav_range, opaci_path)
    bd     = jdi.inputs(calculation="browndwarf")
    bd.phase_angle(0)
    # Convert log g [cgs] to grav [m s^-2]
    gravity = 10**logg * 1e-2   # 1 cm s^-2 = 0.01 m s^-2
    bd.gravity(gravity, gravity_unit=u.Unit('m/s**2'))
    bd.sonora(sonor_path, Teff)

    # Inject corrected TP
    sup.inject_corr(bd, Teff, logg, fsed)

    # Inject Kzz (match pressure grid length)
    prof   = bd.inputs['atmosphere']['profile']
    P      = np.asarray(prof["pressure"], float)
    T      = np.asarray(prof['temperature'], float)
    bd.inputs["atmosphere"]["profile"]["kz"] = [float(kzz)] * len(P)

    # Recommended gas
    rec    = vj.recommend_gas(P, T, MH, MU, plot=False)
    # Remove THESE species (first 2 not supported) (other just choice BD)
    exclude = {'CaAl12O19', 'SiO2', 'CH4', 'Cr', 'TiO2', 'Al2O3', 'CaTiO3', 'ZnS',
               'H2O', 'MnS', 'NH3'} # This row is extra exclude
    # Should only have any combo of these:
    # ['Fe', 'KCl', 'Mg2SiO4', 'MgSiO3', 'Na2S']
    clouds = [sp for sp in rec if sp not in exclude]

    # Saving cloud names, like 123 means clouds 1,2 and 3 used
    cl_names = ''.join(sorted(cloud_dict[c] for c in clouds))

    # Clouds
    bd.virga(clouds, virga_path, fsed, mh=MH, mmw=MU) 

    ######## PC VERSION TO SAVE RAM ########

    # Splitting up the wavelength to run chuncks of wavelengths
    wl_min, wl_max = wav_range
    NCHUNKS        = 25            # bump this up if memory is still tight
    edges          = np.linspace(wl_min, wl_max, NCHUNKS + 1)

    wn_all = []
    fl_all = []

    for a, b in zip(edges[:-1], edges[1:]):
        # tiny pad so boundaries have overlap; avoids empty bins
        eps = 1e-6
        lo  = float(max(wl_min, a) ) - eps
        hi  = float(min(wl_max, b) ) + eps
        if hi <= lo:
            continue

        # Build opacity for this small window and compute spectrum
        opa_chunk = jdi.opannection([lo, hi], opaci_path)
        out = bd.spectrum(opa_chunk, full_output=True)

        wn_chunk = np.asarray(out["wavenumber"])
        fl_chunk = np.asarray(out["thermal"])

        # Skip empty/degenerate returns
        if wn_chunk.ndim != 1 or fl_chunk.ndim != 1:
            del opa_chunk, out
            gc.collect()
            continue
        if wn_chunk.size == 0 or fl_chunk.size == 0:
            del opa_chunk, out
            gc.collect()
            continue

        # Accumulate
        wn_all.append(wn_chunk)
        fl_all.append(fl_chunk)

        del opa_chunk, out, wn_chunk, fl_chunk
        gc.collect()

    # Stitch & sort by wavenumber
    wn = np.concatenate(wn_all)
    fl = np.concatenate(fl_all)
    idx = np.argsort(wn)
    wn, fl = wn[idx], fl[idx]

    # Deduplicate any tiny overlaps (keep first occurrence)
    keep = np.ones_like(wn, dtype=bool)
    keep[1:] = np.abs(np.diff(wn)) > 0
    wn, fl = wn[keep], fl[keep]

    # Regrid in wavenumber space to target R
    wn, fl = jdi.mean_regrid(wn, fl, R=R)

    ######## PC VERSION TO SAVE RAM ########

    return fl, cl_names

In [4]:
##### GENERATE AND SAVE SPECTRUM (MARGE format, single-case files)

Teff_s = [790.3769135035916]      # K
grav_s = [2056.7149450539773]
logg_s = np.log10(np.array(grav_s) * 100)  # log g cgs
fsed_s = [2.0] 
kzz_s  = [1e9]

for Teff_i, logg_i, fsed_i, kzz_i in product(Teff_s, logg_s, fsed_s, kzz_s):

    # Run spectrum
    F_i, names = bd_spectrum(Teff_i, logg_i, fsed_i, kzz_i)
    g          = 10**logg_i * 1e-2   # 1 cm s^-2 = 0.01 m s^-2

    # Filename encodes the parameters; ML will parse from name
    fname  = (f"T{float(Teff_i)}g{float(g)}f{float(fsed_i)}"
              f"{sup.format_kzz(kzz_i)}c{names}.npy")
    fpath  = output_path / fname

    np.save(fpath, F_i)

MemoryError: Unable to allocate 12.5 MiB for an array with shape (90, 18179) and data type float64